# <i> Simple Arsenal + Optuna search

<i>
<i>

Recent updates:

<div style="border: 3px dashed white; padding: 10px; box-sizing: border-box;">

- constants,
- loading the cleaned recordings and creating the interaction features,
- the existing `make_windows` approach,
- an Optuna objective,
- SQLite storage so the study can be resumed.

</div>

<i>
<i>

`cleaned_dataset_3.csv` (sinetänsalmentio_Acc1_m792) is always held out and used to score every trial. All other recordings are used for training.

**Important:** because Optuna repeatedly evaluates configurations on dataset 3, it is acting as a validation/selection set. Its score is not an unbiased final test score so you need to perform that at your own perogative! :)

In [ ]:

import os

# We use this to throttle the number of threads 
# Numbas module is using. Not doing this sometimes
# Makes sktime overuse your processor resources, which
# makes your PC freeze!
os.environ["NUMBA_NUM_THREADS"] = "1" 

from pathlib import Path
import numpy as np
import pandas as pd
import optuna
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import balanced_accuracy_score
from threadpoolctl import threadpool_limits
from sktime.classification.kernel_based import Arsenal

# We're going to set the logging of OPTUNA to on.
# That way we can easily monitor progress as it proceeds. 
optuna.logging.set_verbosity(optuna.logging.INFO)

### <i> CONSTANTS

In [ ]:

# Here is where my data is
DATA_DIR = Path(r"c:\Users\kykis\Documents\Roadscanners\cleaned")

# All my datasets follow this convention
FILE_PATTERN = "cleaned_dataset_{}.csv"

# I use 7 total roadscanners datasets
N_RECORDINGS = 7

TEST_RECORDING = 3 # THIS IS MY TEST RECORDING (sinetänsalmentie M793)
TARGET = "Frostbumps"
ORDER_COL = "Num"

# Fixed feature set from the current Arsenal setup
# I highly recommend dropping some of these that clearly 
# Aren't useful, like Yaw(deg)
FEATURES = [
    "RBCSV_80(cf)",
    "Pitch(deg)",
    "Yaw(deg)",
    "GyrZ",
    "AccY",
    "AccZ_GyrX",
    "AccZ_GyrY",
    "AccZ_GyrZ",
    "AccZ_RBCSV80",
    "AccY_RBCSV80",
    "Acceleration_Magnitude",
    "Gyroscope_Magnitude",
]

# Optuna search ranges
# ADJUST THESE as you see fit.
# NOTE: When resuming trials, it will still remember
# roughly what worked and what didn't.
WINDOW_RANGE = (16, 128)
WINDOW_STEP = 4

STRIDE_RANGE = (5, 40)
STRIDE_STEP = 5

NUM_KERNELS_RANGE = (250, 2000)
NUM_KERNELS_STEP = 250

N_ESTIMATORS_RANGE = (5, 30)
N_ESTIMATORS_STEP = 5

# You could also add multirocket here, but it's very slow.
ROCKET_TRANSFORMS = ["rocket", "minirocket"]

# CPU usage (For when you still need to use your PC while trials are happening.)
ARSENAL_JOBS = 2
CPU_LIMIT = 2

# Increase the number of experiments here to set how long the session will run.
N_TRIALS = 200  
SEED = 42

# SQLite (Optionally, you could use DuckDB too)
STUDY_NAME = "frostbumps_arsenal_simple"
STORAGE = "sqlite:///frostbumps_arsenal_simple.db" 

### <i> LOAD DATA

This section does the following:

1. Loads each dataset
2. Then it adds interaction features. I deliberately left my good ones out.
3. X,y Splits
3. Train, test splits. It does this by creating a list of train recordings (indicated by the recording_id)

In [ ]:

recordings = {}

for recording_id in range(N_RECORDINGS):
    file_path = DATA_DIR / FILE_PATTERN.format(recording_id)

    recording = pd.read_csv(file_path, index_col=0)

    # Create interaction features
    recording["AccZ_GyrX"] = recording["AccZ"] * recording["GyrX"]


    # INSERT MORE INTERACTION FEATURES HERE AS SO.

    recording = recording.sort_values(ORDER_COL)

    # We split the featres and the target (and may as well)
    # switch it to numpy now, since that's what Arsenal/Keras
    # expects.
    X = recording[FEATURES].to_numpy(dtype=np.float32)
    y = recording[TARGET].to_numpy(dtype=int)

    # Make the recording_id column.
    recordings[recording_id] = (X, y)

# List comprehension that appends all datasets not called
# the test recording.
TRAIN_RECORDINGS = [
    recording_id
    for recording_id in recordings
    if recording_id != TEST_RECORDING
]

# And here is the final layout for the test/train split.
print("Training recordings:", TRAIN_RECORDINGS)
print("Testing recording:", TEST_RECORDING)

Training recordings: [0, 1, 2, 4, 5, 6]
Testing recording: 3


### <i> Make windows

This is a really minimal windowing function. `recording_id` either takes a list of recording_ids or just one recording_id.

It has been tested for dataleaks 

In [ ]:

def make_windows(recording_ids, window_size, stride):
    X_windows = []
    y_windows = []

    for recording_id in recording_ids:
        X, y = recordings[recording_id]

        if len(X) < window_size:
            continue

        # Shape: (samples, features, timesteps)
        X_temp = sliding_window_view(X,window_shape=window_size,axis=0)[::stride]

        # Label each window using its final timestep
        y_temp = y[window_size - 1::stride]

        # Adding the windows to the window lists above.
        X_windows.append(np.array(X_temp))
        y_windows.append(y_temp)

    X_windows = np.concatenate(X_windows, axis=0)
    y_windows = np.concatenate(y_windows)

    return X_windows, y_windows

### <i> OPTUNA objective

<i> This runs the main trial with the constant suggestions above. 

In [ ]:


def objective(trial):

    # This runs your suggestion parameters (which were constants in the cells above)
    window_size = trial.suggest_int(
        "window_size",
        *WINDOW_RANGE,
        step=WINDOW_STEP
    )

    stride = trial.suggest_int(
        "stride",
        *STRIDE_RANGE,
        step=STRIDE_STEP
    )

    num_kernels = trial.suggest_int(
        "num_kernels",
        *NUM_KERNELS_RANGE,
        step=NUM_KERNELS_STEP
    )

    n_estimators = trial.suggest_int(
        "n_estimators",
        *N_ESTIMATORS_RANGE,
        step=N_ESTIMATORS_STEP
    )

    rocket_transform = trial.suggest_categorical(
        "rocket_transform",
        ROCKET_TRANSFORMS
    )

    # Create training and testing windows
    X_train, y_train = make_windows(TRAIN_RECORDINGS, window_size, stride)
    X_test, y_test = make_windows([TEST_RECORDING], window_size, stride)

    # Train Arsenal
    model = Arsenal(
        num_kernels=num_kernels,
        n_estimators=n_estimators,
        rocket_transform=rocket_transform,
        n_jobs=ARSENAL_JOBS,
        random_state=SEED
    )

    # We're throttling our CPU this way so it can run stable long term.
    # I highly recommend keeping your CPU_LIMIT lower than your CPU specs.
    # You have been warned. 
    with threadpool_limits(limits=CPU_LIMIT):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Score the trial
    score = balanced_accuracy_score(y_test, y_pred)

    return score

### <i> RUN / RESUME OPTUNA

In [ ]:
# Run / Interrupt this cell freely as needed. 
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

with threadpool_limits(limits=CPU_LIMIT):
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        gc_after_trial=True
    )

print("\\nBest balanced accuracy:", study.best_value)
print("Best parameters:")

for parameter, value in study.best_params.items():
    print(f"{parameter}: {value}")

[I 2026-11-22 01:16:07,792] Using an existing study with `study_name='frostbumps_arsenal_simple'` instead of creating a new one.
[I 2026-11-22 01:17:58,827] Trial 13 finished with value: 0.6089108910891089 and parameters: {'window_size': 116, 'stride': 20, 'num_kernels': 1750, 'n_estimators': 10, 'rocket_transform': 'minirocket'}. Best is trial 8 with value: 0.8845478489903424.
[I 2026-11-22 01:18:19,985] Trial 14 finished with value: 0.85625 and parameters: {'window_size': 52, 'stride': 25, 'num_kernels': 1250, 'n_estimators': 5, 'rocket_transform': 'minirocket'}. Best is trial 8 with value: 0.8845478489903424.


---

<i>

## Viewing previous trials

The SQLite database is updated automatically. Re-running the Optuna cell loads the existing study and adds `N_TRIALS` more trials.

You can inspect the results with:

In [ ]:

results = study.trials_dataframe(
    attrs=("number", "value", "params", "state")
)

results = results.sort_values("value", ascending=False)

display(results.head(10))